# PySpark Basic DataFrame Operations

**Learning PySpark fundamentals through hands-on DataFrame operations with Apple stock data.**

## Project Overview

This project teaches PySpark DataFrame basics using Apple stock price data. We cover creating SparkSessions, reading CSVs, schema inspection, selecting/filtering/aggregating data, and SQL queries — the essential building blocks for big data processing.

## Learning Objectives

1. Create and configure a SparkSession
2. Read CSV data into PySpark DataFrames
3. Inspect schema, data types, and basic statistics
4. Perform select, filter, groupBy, and aggregate operations
5. Use PySpark SQL for querying DataFrames
6. Understand when PySpark is appropriate vs pandas

## Business / Research Problem

As data grows beyond single-machine capacity, PySpark enables distributed processing. Understanding DataFrame operations is foundational for:
- **Data engineers** building ETL pipelines
- **Data scientists** working with large datasets
- **Analysts** querying distributed data

**Key question:** *How do we perform common data operations using PySpark's distributed DataFrame API?*

## Why This Analysis Matters

PySpark is the industry standard for big data processing. Learning its DataFrame API prepares you for real-world data engineering and large-scale analytics roles.

## Dataset Overview

Apple stock price data with columns: `Date`, `Open`, `High`, `Low`, `Close`, `Volume`, `Adj. Close`, etc.

**File:** `apple.csv` in the project directory

## Dataset Source & License Notes

- **Source:** Historical stock data
- **License:** Public financial data
- **File:** Local `apple.csv`

## Environment Setup

In [ ]:
!pip install pyspark pandas numpy matplotlib -q

## Imports

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, mean, max, min, count, stddev, year, month, dayofmonth
from pyspark.sql.functions import format_number, corr
from pyspark.sql.types import *
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
print('All imports loaded successfully.')

## Configuration / Constants

In [ ]:
DATA_FILE = 'apple.csv'
APP_NAME = 'PySpark_DataFrame_Basics'

## Dataset Download / Loading

Create a SparkSession and load the CSV data.

In [ ]:
# Create SparkSession
spark = SparkSession.builder.appName(APP_NAME).getOrCreate()
print(f'Spark version: {spark.version}')

# Read CSV with header and infer schema
df = spark.read.csv(DATA_FILE, header=True, inferSchema=True)
print(f'Loaded {df.count()} rows')
df.show(5)

## Data Validation Checks

Inspect schema, data types, and basic statistics.

In [ ]:
# Schema
print('=== Schema ===')
df.printSchema()

# Column names
print(f'\nColumns: {df.columns}')
print(f'Row count: {df.count()}')

In [ ]:
# Describe (equivalent to pandas describe)
df.describe().show()

## Data Cleaning

Handle any nulls and ensure proper data types.

In [ ]:
# Check for nulls
from pyspark.sql.functions import col, sum as spark_sum

null_counts = df.select([spark_sum(col(c).isNull().cast('int')).alias(c) for c in df.columns])
print('=== Null Counts ===')
null_counts.show()

# Drop rows with nulls if any
initial = df.count()
df = df.dropna()
print(f'Rows after dropping nulls: {df.count()} (dropped {initial - df.count()})')

## Exploratory Data Analysis

### Basic DataFrame Operations

In [ ]:
# Select specific columns
print('=== Select Close and Volume ===')
df.select('Close', 'Volume').show(5)

# Select with formatting
print('=== Formatted Output ===')
df.select('Date', format_number('Close', 2).alias('Close'),
          format_number('Volume', 0).alias('Volume')).show(5)

In [ ]:
# Filter operations
print('=== Days where Close > Open (price went up) ===')
df.filter(df['Close'] > df['Open']).select('Date', 'Open', 'Close').show(5)

# Count up vs down days
up_days = df.filter(df['Close'] > df['Open']).count()
down_days = df.filter(df['Close'] <= df['Open']).count()
print(f'Up days: {up_days}, Down days: {down_days}')

## Univariate Analysis

In [ ]:
# Summary statistics for Close price
print('=== Close Price Statistics ===')
df.select(
    mean('Close').alias('Mean'),
    min('Close').alias('Min'),
    max('Close').alias('Max'),
    stddev('Close').alias('StdDev'),
    count('Close').alias('Count')
).show()

In [ ]:
# Convert to pandas for visualization
pdf = df.toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(pdf['Close'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Close Price Distribution')
axes[0].set_xlabel('Close Price')

axes[1].hist(pdf['Volume'], bins=30, color='coral', edgecolor='white')
axes[1].set_title('Volume Distribution')
axes[1].set_xlabel('Volume')

plt.tight_layout()
plt.show()

## Bivariate / Multivariate Analysis

In [ ]:
# Correlation between High and Volume
print('=== Correlations ===')
df.select(corr('High', 'Volume').alias('High_Volume_Corr')).show()
df.select(corr('Open', 'Close').alias('Open_Close_Corr')).show()

In [ ]:
# Time series plot
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(range(len(pdf)), pdf['Close'], alpha=0.7, color='steelblue', linewidth=0.5)
ax.set_title('Apple Stock Close Price Over Time')
ax.set_xlabel('Trading Day')
ax.set_ylabel('Close Price')
plt.tight_layout()
plt.show()

## Feature-Specific Insights

### GroupBy and Aggregations

In [ ]:
# Add year column for aggregation
df_year = df.withColumn('Year', year(col('Date')))

# Yearly statistics
print('=== Yearly Average Close Price ===')
df_year.groupBy('Year').agg(
    format_number(mean('Close'), 2).alias('Avg_Close'),
    format_number(mean('Volume'), 0).alias('Avg_Volume'),
    count('*').alias('Trading_Days')
).orderBy('Year').show(20)

## Statistical Checks / Hypothesis Testing

### Using Spark SQL

In [ ]:
# Register as temp table
df.createOrReplaceTempView('apple_stock')

# SQL query example
print('=== Top 5 Highest Close Prices (SQL) ===')
spark.sql('SELECT Date, Close, Volume FROM apple_stock ORDER BY Close DESC LIMIT 5').show()

print('=== SQL Aggregate ===')
spark.sql('''
    SELECT 
        COUNT(*) as total_days,
        ROUND(AVG(Close), 2) as avg_close,
        ROUND(AVG(Volume), 0) as avg_volume
    FROM apple_stock
''').show()

## Visual Analysis

In [ ]:
# Daily price range
pdf['Range'] = pdf['High'] - pdf['Low']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(pdf['Volume'], pdf['Close'], alpha=0.3, s=5, c='steelblue')
axes[0].set_title('Volume vs Close Price')
axes[0].set_xlabel('Volume')
axes[0].set_ylabel('Close')

axes[1].hist(pdf['Range'], bins=30, color='mediumseagreen', edgecolor='white')
axes[1].set_title('Daily Price Range (High - Low)')
axes[1].set_xlabel('Range')

plt.tight_layout()
plt.show()

## Key Findings

1. **PySpark DataFrames mirror SQL operations** — select, filter, groupBy, aggregate
2. **Schema inference works well** for clean CSV data
3. **Spark SQL provides a familiar interface** for analysts coming from SQL backgrounds
4. **GroupBy + aggregation** is the core pattern for summarizing distributed data
5. **toPandas() bridges Spark and local visualization** — useful but only for results that fit in memory

## Limitations

1. **Small dataset:** This dataset fits in pandas — PySpark overhead isn't justified here
2. **Local mode only:** Running on a single machine misses PySpark's distributed benefits
3. **No streaming:** This covers batch processing only
4. **Limited ML:** DataFrame operations only — no MLlib in this notebook

## Recommendations / Next Steps

1. Try PySpark on a larger dataset (>1GB) to see real performance benefits
2. Explore PySpark MLlib for distributed machine learning
3. Learn window functions for time-series operations

### How to Extend This Analysis
- Add moving averages using PySpark window functions
- Join multiple stock datasets
- Build a PySpark streaming pipeline for real-time stock data

## Common Mistakes

1. **Using PySpark for small data:** Pandas is faster for datasets under ~1GB
2. **Calling toPandas() on large DataFrames:** This collects ALL data to the driver — will crash on big data
3. **Not caching frequently used DataFrames:** Use `.cache()` to avoid recomputation
4. **Forgetting lazy evaluation:** Transformations aren't executed until an action (show, count, collect) is called
5. **Using Python UDFs instead of built-in functions:** Built-in PySpark functions are much faster

## Mini Challenge / Exercises

1. What day had the highest single-day price increase (Close - Open)?
2. Calculate a 30-day moving average of Close price using window functions
3. Write a SQL query to find months with average volume above the overall average
4. Create a new column `daily_return` = (Close - Open) / Open * 100
5. Find the top 10 highest-volume trading days

In [ ]:
# Space for exercise solutions
# Exercise 1: df.withColumn('gain', col('Close') - col('Open')).orderBy(col('gain').desc()).show(1)
# Exercise 5: df.orderBy(col('Volume').desc()).show(10)

In [ ]:
# Clean up
spark.stop()
print('SparkSession stopped.')

## Final Summary / Key Takeaways

- **PySpark DataFrames are the distributed equivalent of pandas DataFrames**
- **Core operations:** select, filter, groupBy, agg, orderBy, withColumn
- **Spark SQL** lets you query DataFrames with familiar SQL syntax
- **Always prefer built-in functions** over Python UDFs for performance
- **Use toPandas() sparingly** — only for small result sets destined for visualization
- **PySpark shines on large datasets** — for small data, stick with pandas

Master these basics and you're ready for production-scale data engineering.